# Advanced Parameter Sampling

This tutorial builds off the "Sampling Parameters" notebook to describe how to implement more complex dependencies between parameters. It includes numerous examples that users can use as a template for their own approaches. Users do not need to read the whole thing from top to bottom. Feel free to jump to different sections, picking and choosing the techniques you explore.

## Core Concepts

In the ["Sampling Parameters" notebook](https://lightcurvelynx.readthedocs.io/en/latest/notebooks/sampling.html), we introduced a few core concepts that will be heavily used throughout this notebook and are worth reviewing:

* A *parameter* is effectively a variable in the mathematical equations that is instantiated during a round of sampling. A simple model might contain a handful of parameters for everything from position on the sky (RA, Dec) to inherent physical quantities (hostmass) to purely functional parameters (curve decay rate). 
* A `ParameterizedNode` is a computational unit for working with parameters. These nodes provide the code that defines the recipe for computing its own parameters. These nodes may use inputs that are parameters computed within other nodes.
* LightCurveLynx treats parameter settings (when constructing the object) as linking the parameter to a **recipe** for dynamically instantiating concrete values of that variable 
* A `GraphState` is a data structure that holds the sampled values for all the parameters in the model. Each `ParameterizedNode` object is stateless and does not store information about the parameters themselves. Instead all operations take a `GraphState` that contains the necessary input parameters and will store the corresponding output parameters.

The combined values of **all** the parameters in the graph define a single sample of the model's parameters.

## Basic Chaining

We can use one `ParameterizedNode` object to provide parameters for another object. As described in the previous notebook, this chaining can use the output of a node as the input to another node:

In [ ]:
import numpy as np

from lightcurvelynx.math_nodes.np_random import NumpyRandomFunc
from lightcurvelynx.models.basic_models import ConstantSEDModel

# Define a brightness distribution and create a ConstantSEDModel that samples from it.
brightness_dist = NumpyRandomFunc("normal", loc=20.0, scale=2.0, node_label="brightness_dist")
model = ConstantSEDModel(brightness=brightness_dist, node_label="test")
state = model.sample_parameters(num_samples=10)
print(state["test"]["brightness"])

In this example, the value of the object's brightness parameter is dynamically generated using the recipe provided by the `brightness_dist` node. Since that node is generating samples from a normal distribution, the object's brightness is being sampled as a normal distribution.

We can also reference another node's parameter using the dot notation. For example, if we want to access the host's position information we could use "<host_object>.ra" and "<host_object.dec>":

In [ ]:
ra_node = NumpyRandomFunc("uniform", low=0.0, high=360.0)
host = ConstantSEDModel(brightness=15.0, ra=ra_node, dec=2.0, node_label="host")
source = ConstantSEDModel(brightness=10.0, ra=host.ra, dec=host.dec, node_label="source")
state = source.sample_parameters(num_samples=10)
for i in range(10):
    print(
        f"Sample {i + 1}: Host RA = {state['host']['ra'][i]:.2f}, Source RA = {state['source']['ra'][i]:.2f}"
    )

When a parameter is assigned from another node using the dot notation, the user is telling the code to link the parameters. In other words, the recipe for `source.ra` is "Copy the value from `host.ra`." When the host samples a new value for RA, it is effectively passed along to the source.

## Parameter Collections

In some cases, users may want to define a single node that implements a collection of related parameters. For example, we may want to have a single node that samples all the SNIa parameters from a single paper. We can conveniently group parameters by defining them in a single subclass of `ParameterizedNode`.

Consider this example, using settings from the ["Basic SN Ia Example" notebook](https://lightcurvelynx.readthedocs.io/en/latest/notebooks/simple_snia.html):

In [ ]:
from lightcurvelynx.astro_utils.snia_utils import DistModFromRedshift, X0FromDistMod
from lightcurvelynx.base_models import ParameterizedNode
from lightcurvelynx.math_nodes.np_random import NumpyRandomFunc


class MySNIAParameters(ParameterizedNode):
    """Sampler for SNIa parameters."""

    def __init__(self, **kwargs):
        # Handle the initialization of the parameterized node.
        super().__init__(**kwargs)

        # Sample redshift uniformly between 0.01 and 0.2.
        self._redshift_sampler = NumpyRandomFunc("uniform", low=0.01, high=0.2)
        self.add_parameter("redshift", self._redshift_sampler)

        # Define the distmod function
        self._distmod_func = DistModFromRedshift(self._redshift_sampler, H0=73.0, Omega_m=0.3)
        self.add_parameter("distmod", self._distmod_func)

        # Sample x1, c, and m_abs from distributions motivated by typical SNIa values.
        self._x1_func = NumpyRandomFunc("normal", loc=0, scale=2.0)
        self._c_func = NumpyRandomFunc("normal", loc=0, scale=0.02)
        self._m_abs_func = NumpyRandomFunc("normal", loc=-19.3, scale=0.1)
        self.add_parameter("x1", self._x1_func)
        self.add_parameter("c", self._c_func)
        self.add_parameter("m_abs", self._m_abs_func)

        # Compute x0 from the other parameters using the standard Tripp formula,
        # and the distance modulus from above.
        self._x0_func = X0FromDistMod(
            distmod=self._distmod_func,
            x1=self._x1_func,
            c=self._c_func,
            alpha=0.14,
            beta=3.1,
            m_abs=self._m_abs_func,
        )
        self.add_parameter("x0", self._x0_func)


snia_sampler = MySNIAParameters()
print(snia_sampler.sample_parameters(num_samples=5))

As we saw in the ["Basic SN Ia Example" notebook](https://lightcurvelynx.readthedocs.io/en/latest/notebooks/simple_snia.html), these parameters can all be individually defined when setting up the object. However, by wrapping them in a single node, we provide a convenience wrapper. Users can use this combined distribution with a single line.

Each individual attribute can be accessed using the dot notation.

In [ ]:
from lightcurvelynx.models.sncosmo_models import SncosmoWrapperModel

source = SncosmoWrapperModel(
    "salt2-h17",  # Model name
    x0=snia_sampler.x0,
    x1=snia_sampler.x1,
    c=snia_sampler.c,
    redshift=snia_sampler.redshift,
    ra=150.0,
    dec=-30.0,
    t0=0.0,
    node_label="source",
)

print(source.sample_parameters(num_samples=5))

## Sampling from Known Values

LightCurveLynx provides multiple `ParameterizedNode` subclasses for selecting known values in math_nodes/given_sampler.py. These can be used for testing, allowing a user to input different (but known) values for each sample. As we will see later in this notebook, they can also be combined with other node types to do more complex computations (see the "Combining Node Types" and "Sample Based Distributions" sections below for examples).

### BinarySampler

The `BinarySampler` node returns a single ``True`` or ``False`` value for each sample. This is specifically designed for is probabilistically applying effects or making decisions in the simulation.

In [ ]:
from lightcurvelynx.math_nodes.given_sampler import BinarySampler

apply_effect = BinarySampler(0.25, node_label="apply_effect")
states = apply_effect.sample_parameters(num_samples=5000)

num_true = np.count_nonzero(states["apply_effect"]["function_node_result"])
print(f"Returned {num_true} TRUE and {5000 - num_true} FALSE")

### GivenValueList

The `GivenValueList` node returns the values from a given list (in the order in which they are given). This is primarily used for testing:

In [ ]:
from lightcurvelynx.math_nodes.given_sampler import GivenValueList

brightness_dist = GivenValueList([18.0, 20.0, 22.0, 25.0], stateful=True)
model = ConstantSEDModel(brightness=brightness_dist, node_label="test")
state = model.sample_parameters(num_samples=3)
print(state["test"]["brightness"])

The ``GivenValueList`` is the only stateful parameterized node (when the ``stateful`` parameter is set to ``True``). For testing purposes if you query the node multiple times, it will give the next unsampled items from the list.

In [ ]:
state = model.sample_parameters(num_samples=1)
print(state["test"]["brightness"])

If ``stateful=False`` is used the sampler will restart at the beginning of the list for each batch.

Note that, when stateful, the `GivenValueList` does **not** support parallel execution. The simulation will fail with an error if a `GivenValueList` with ``stateful=True`` is used.

### GivenValueSampler

The `GivenValueSampler` node returns a random value (with replacement) from a given list:

In [ ]:
from lightcurvelynx.math_nodes.given_sampler import GivenValueSampler

brightness_dist = GivenValueSampler([18.0, 20.0, 22.0])
model = ConstantSEDModel(brightness=brightness_dist, node_label="test")
state = model.sample_parameters(num_samples=10)
print(state["test"]["brightness"])

The node can also take a list of weights to sample from different distributions.

In [ ]:
brightness_dist = GivenValueSampler([18.0, 20.0, 22.0], weights=[0.5, 0.3, 0.2])
model = ConstantSEDModel(brightness=brightness_dist, node_label="test")
state = model.sample_parameters(num_samples=1000)
print("Brightness 18:", np.count_nonzero(state["test"]["brightness"] == 18.0))
print("Brightness 20:", np.count_nonzero(state["test"]["brightness"] == 20.0))
print("Brightness 22:", np.count_nonzero(state["test"]["brightness"] == 22.0))

We can also provide a single integer value N to sample from the range [0, N-1].

In [ ]:
brightness_dist = GivenValueSampler(10)
model = ConstantSEDModel(brightness=brightness_dist, node_label="test")
state = model.sample_parameters(num_samples=1000)
for i in range(5):
    print(f"Brightness {i}:", np.count_nonzero(state["test"]["brightness"] == i))

This is useful when sampling indices of other lists (see the example in the "Combining Node Types" section below).

### GivenValueSelector

The `GivenValueSelector` node takes a single input parameter *index* and uses that to lookup the parameter's value from a given list. Which item is selected is determined by the `GivenValueSelector`'s *index* parameter. If we sample the *index* from a given distribution we can sample items for the list.

Below we use a constant value for *index* so we return the same element each time:

In [ ]:
from lightcurvelynx.math_nodes.given_sampler import GivenValueSelector

brightness_dist = GivenValueSelector([18.0, 20.0, 22.0], index=2)
model = ConstantSEDModel(brightness=brightness_dist, node_label="test")
state = model.sample_parameters(num_samples=10)
print(state["test"]["brightness"])

At first glance, the `GivenValueSelector` node may appear redundant with the `GivenValueSampler` node. After all, both nodes are selecting a value from a list. However, as we will see in the next section, the power of these nodes comes from when they are used together to sample consistently from multiple lists.

## Combining Node Types

We can perform complex sampling operations by combining multiple types of nodes. For example, imagine that we wanted to sample from a list of known objects where we have a list of the RAs, decs, brightness, and redshifts. We can combine a random selection of the object's index (by sampling a `index` parameter) with nodes that look up the value for that object index in each of the corresponding lists:

In [ ]:
ra_list = [10.0, 20.0, 30.0, 40.0, 50.0]
dec_list = [1.0, 2.0, 3.0, 4.0, 5.0]
brightness_list = [15.0, 16.0, 17.0, 18.0, 19.0]

index_dist = GivenValueSampler(5)  # Samples indices 0 to 4 uniformly

# Use the same sampled index for ra, dec, and brightness.
model = ConstantSEDModel(
    brightness=GivenValueSelector(brightness_list, index=index_dist),
    ra=GivenValueSelector(ra_list, index=index_dist),
    dec=GivenValueSelector(dec_list, index=index_dist),
    node_label="model",
)

state = model.sample_parameters(num_samples=10)
for i in range(10):
    ra = state["model"]["ra"][i]
    dec = state["model"]["dec"][i]
    brightness = state["model"]["brightness"][i]
    print(f"Sample {i + 1}: ({ra}, {dec}) = {brightness}")

The `GivenValueSampler` node chooses an object index value from the range [0, 5). The output of this node (the index) is passed as the input to multiple `GivenValueSelector` nodes to extract the corresponding element from each of the lists.

Any important consideration is that each node in the graph is only sampled once. This means a *single* index is chosen and used for all three lists. For each sample, the value of all parameters (RA, Dec, and brightness) will be consistent for a single object.

For other examples of how these types of nodes can be combined, see the implementation of the `MultiLightcurveTemplateModel` and the `RandomMultiObjectModel` models.

## Sampling from Tables

Instead of lists, we might want to extract values from tabular data represented as a dictionary, AstroPy Table, or Pandas Dataframe. The `TableSampler` node will sample a row from given tabular data and store a unique parameter for each column of the table.

For example, we can create a table columns 'A', 'B', and 'C' and sample from those:

In [ ]:
from astropy.table import Table

from lightcurvelynx.math_nodes.given_sampler import TableSampler

raw_data_dict = {
    "A": [1, 2, 3, 4, 5, 6, 7, 8],
    "B": [2, 3, 4, 5, 4, 3, 2, 1],
    "C": [3, 4, 5, 6, 7, 8, 9, 10],
}
data = Table(raw_data_dict)

table_node = TableSampler(data, in_order=True, node_label="node")
state = table_node.sample_parameters(num_samples=3)
print(state)

The `in_order` flag tells the node whether to extract the rows in order (`True`) or randomly with replacement (`False`). Note that the `TableSampler` is **not** stateful. If called multiple times (with `in_order=True`), it will return the first N rows from the table each time. It will also produce a warning for the user.

In [ ]:
state = table_node.sample_parameters(num_samples=3)
print(state)

If users want to perform multiple sequential simulations using different parts of the table, then they will need to use the ``sample_offset`` parameter.

In [ ]:
state = table_node.sample_parameters(num_samples=3, sample_offset=3)
print(state)

However, most users should not need to ever set ``sample_offset`` directly. If you are running a parallelized simulation, the software sets and uses this value behind the scenes to ensure that every worker is operating on a different part of the table.

As with other node types, we can use the dot notation to use these values as input for other models. For example, let’s assume that the 'B' column corresponds to Brightness, 'A' corresponds to RA, and 'C' is not used.

In [ ]:
table_node = TableSampler(data, in_order=False, node_label="node")
model = ConstantSEDModel(
    brightness=table_node.B,
    ra=table_node.A,
    node_label="test",
)

state = model.sample_parameters(num_samples=10)
print(state)

## Statistical Distributions

### Basic (Numpy) Distributions

The `NumpyRandomFunc` node allows users to sample from most simple distributions supported by numpy, such as uniform or normal. The node's constructor takes the name of the numpy function and then any parameters as keyword arguments.

In the following cell we generate 10,000 samples from a normal distribution with mean=0.0 and standard deviation=1.0.

In [ ]:
import matplotlib.pyplot as plt

normal_dist = NumpyRandomFunc("normal", loc=0.0, scale=1.0, node_label="normal_dist")
samples = normal_dist.sample_parameters(num_samples=10000)
plt.hist(samples["normal_dist"]["function_node_result"], bins=50, density=True)
plt.title("Histogram of Samples from Normal Distribution")
plt.xlabel("Value")
plt.ylabel("Density")
plt.show()

The keyword arguments for the `NumpyRandomFunc` can be chained to take other parameters themselves. In the following example, we generate an x value for each row uniformly [1, 10]. Then we use that x value to set the mean of the normal distribution for that row.

In [ ]:
x_dist = NumpyRandomFunc("uniform", low=1.0, high=10.0, node_label="mean_dist")
normal_dist = NumpyRandomFunc("normal", loc=x_dist, scale=1.0, node_label="normal_dist")
samples = normal_dist.sample_parameters(num_samples=1_000)

plt.scatter(
    samples["mean_dist"]["function_node_result"], samples["normal_dist"]["function_node_result"], alpha=0.5
)
plt.xlabel("Mean")
plt.ylabel("Value")
plt.title("Scatter Plot of Samples from Normal Distribution with Varying Mean")
plt.show()

### Sample Based Distributions

We can combine the known value samplers with basic statistical functions to produce more complex statistical distributions. For example, consider modeling a multi-model distribution:

In [ ]:
centers1 = np.random.normal(5.0, 0.5, 200)
centers2 = np.random.normal(10.0, 0.5, 50)
all_centers = np.concatenate([centers1, centers2])

_ = plt.hist(all_centers, bins=50, density=True)
plt.title("Histogram of Samples from Normal Distribution")
plt.xlabel("Value")
plt.ylabel("Density")
plt.show()

We could model this with a variety of hand-crafted distributions or a sample-based distribution.

We can approximate the samples that we would get from a kernel density estimator by constructing a multi-stage process. First, we choose one of the data points and treat this as the center of our kernel. Second, we sample an offset using a Gaussian distribution. Finally, we compute the sampled point by adding the center and the offset, using the ``BasicMathNode`` introduced in the [sampling parameters notebook](https://lightcurvelynx.readthedocs.io/en/latest/notebooks/sampling.html).

In [ ]:
from lightcurvelynx.math_nodes.basic_math_node import BasicMathNode

center_sampler = GivenValueSampler(all_centers)
offset_sampler = NumpyRandomFunc("normal", loc=0.0, scale=0.05)
value_sampler = BasicMathNode(
    "center + offset",
    center=center_sampler,
    offset=offset_sampler,
    node_label="value_sampler",
)

samples = value_sampler.sample_parameters(num_samples=1_000)

_ = plt.hist(samples["value_sampler"]["function_node_result"], bins=50, density=True)
plt.title("Histogram of Samples from Our Custom Sampler")
plt.xlabel("Value")
plt.ylabel("Density")
plt.show()

### Combining Distributions

There are plenty of times you might want to move beyond the simple numpy sampling functions. For example, a user might want to generate numbers uniformly from the union of ranges [0, 4] U [6, 10].  We can do this by chaining nodes. Here we use a `RandomChoiceNode` (will be introduced in v0.4.2) to pick the distribution.

In [ ]:
from lightcurvelynx.math_nodes.random_choice import RandomChoiceNode

input1 = NumpyRandomFunc("uniform", low=0, high=4, node_label="input1")
input2 = NumpyRandomFunc("uniform", low=6, high=10, node_label="input2")
choice_node = RandomChoiceNode([input1, input2], node_label="choice_node")

sampled_state = choice_node.sample_parameters(num_samples=5_000)
results = sampled_state["choice_node"]["function_node_result"]

plt.hist(results, bins=50)
plt.title("Histogram of Samples from RandomChoiceNode")
plt.xlabel("Value")
plt.show()